> **What this notebook is:** a survivorship-bias / PIT-correction quantification on the legacy v2 22-feature LightGBM panel. The six-cell ablation matrix shows how much apparent IC comes from looking only at today's S&P 500 survivors vs the historical PIT-correct universe, and how the inflation interacts with feature-set strength.
>
> **What this notebook is NOT:** verification of the current README headline. The README's current primary finding is the **9-feature L1 regression at 21-day horizon (`lasso_elasso_pit_h21`): IC +0.0695, t = +5.25, L/S Sharpe +1.24** on the 2025-01-02 → 2026-04-29 OOS slice. To verify the current headline directly, run the apples-to-apples comparison scripts (no notebook needed):
>
> ```bash
> PYTHONPATH=src python scripts/compare_apples_to_apples.py --since 2025-01-01 --horizon 21
> PYTHONPATH=src python scripts/compare_net_of_cost.py --since 2025-01-01 --horizon 21
> ```
>
> The PIT mechanics, deflated-Sharpe correction, and walk-forward harness this notebook documents all apply to the current headline as well — the survivorship-bias caveats from this notebook's results carry through to every model in the README.


# 00 — PIT ablation study: quantifying survivorship bias on the legacy v2 panel

**Why this notebook matters:** survivorship bias is the single biggest distortion in a naive backtest. Today's S&P 500 list is a 30-year survivor cohort — companies that grew, didn't go bankrupt, weren't acquired into oblivion. Backtesting on today's list silently rewards the model for picking 2026 winners *in 2018*, which the model couldn't actually have done.

This notebook quantifies that bias on the project's legacy v2 LightGBM panel (22 OHLCV / anomaly features at 5-day horizon — the earlier primary estimate before the L1 regression became the headline). The six checks below verify a 2×3 ablation matrix that isolates three confounders:

1. **Universe choice** — modern-survivor `sp500.txt` (160 tickers, PIT off) vs PIT-correct `sp500_pit.txt` (617 tickers, PIT on)
2. **Feature-set strength** — 13-feature technical baseline → 16-feature anomaly addition → 22-feature OHLCV/volume addition
3. **Regime split** — full-sample vs post-October-2022 regime cut

| Check | Experiment ID | Expected IC | Expected Sharpe | What it tests |
|---|---|---|---|---|
| Stage 1 (13 features, subset, PIT off) | `extended_kaggle_v2` | **+0.0075** | +0.28 | Historical baseline (weak features, survivor universe) |
| Stage 2 (13 features, PIT on) | `extended_kaggle_v2_pit` | **+0.0008** | -0.04 | 89% IC collapse on weak features under PIT |
| Stage 3 (16 features, PIT on) | `extended_kaggle_v2_anomaly` | **+0.0033** | +0.083 | + 3 academic anomalies on PIT |
| v2 baseline (22 features, PIT on) | `extended_kaggle_v2_ohlcv` | **+0.0055** | +0.21 | + 6 OHLCV/volume on PIT, full sample |
| PIT-off ablation (22 features, subset) | `extended_kaggle_v2_ohlcv_subset` | **+0.0142** | +0.39 | Survivorship cost on strong features (61% inflation) |
| v2 regime split (22 features, PIT on, post-Oct-2022) | (slice of v2_ohlcv) | **+0.0183** | **+0.86** | Legacy primary — regime-conditional residual edge |

**Reading the matrix:**

- *Stage 1 → Stage 2* (rows 1→2): 89% IC drop is the survivorship-bias cost on a *weak* feature set. The 13-feature technical panel barely beats noise on the real (PIT-correct) universe.
- *PIT-off ablation → v2 baseline* (rows 5→4): 61% IC drop is the survivorship-bias cost on the *strongest* feature set this notebook covers. Bias is smaller when signal is stronger (the inflation gets diluted) but still material.
- *v2 baseline → v2 regime split* (rows 4→6): the residual +0.0183 IC concentrates in the post-October-2022 regime. The same model on the pre-cutoff slice produces IC near zero (or slightly negative). The legacy primary was a regime-conditional finding.

These same survivorship caveats apply uniformly to every IC reported in the README, including the current L1 regression headline of +0.0695. The deflated-Sharpe correction, walk-forward harness, and PIT-membership scraper this notebook relies on are exactly the same machinery used by the current headline experiments.

## How to use this notebook

1. Run all five CLI experiments first (prerequisites checklist below). Each writes predictions to the DuckDB store at `artifacts/predictions/predictions.duckdb`.
2. Run all cells of this notebook in order.
3. The final cell prints **PASS / DRIFT / FAIL per check**. Small drift (within ±5% of the quoted IC) is normal and counted as PASS.

## Prerequisites checklist

```bash
pip install -e ".[dev,classical]"
python -m price_model.cli build-universe --name sp500_pit --start 2017-01-01
python -m price_model.cli refresh-data --universe sp500   --start 2017-01-01
python -m price_model.cli refresh-data --universe sp500_pit --start 2017-01-01
python -m price_model.cli run -e extended_kaggle_v2
python -m price_model.cli run -e extended_kaggle_v2_pit
python -m price_model.cli run -e extended_kaggle_v2_anomaly
python -m price_model.cli run -e extended_kaggle_v2_ohlcv
python -m price_model.cli run -e extended_kaggle_v2_ohlcv_subset
```

Cumulative wall-clock: ~90-120 minutes cold, ~15 minutes warm.


In [1]:
from __future__ import annotations

from datetime import date

import polars as pl

from price_model.serving.store import PredictionStore
from price_model.data.loaders import load_panel
from price_model.features.targets import add_forward_excess_return
from price_model.eval.metrics import summarize
from price_model.eval.robustness import time_split_evaluate

# Expected README headlines, with tolerance bands.
EXPECTED = {
    "stage_1_naive": {
        "experiment":   "extended_kaggle_v2",
        "model_id":     "lightgbm_kaggle_v2",
        "universe":     "sp500",
        "pit_filter":   False,
        "ic_expected":  +0.0075,
        "ic_tolerance": 0.0040,
        "sharpe_expected": +0.28,
        "sharpe_tolerance": 0.20,
        "description": "13 features, subset, no PIT — chronological Stage 1 (historical)",
    },
    "stage_2_pit": {
        "experiment":   "extended_kaggle_v2_pit",
        "model_id":     "lightgbm_kaggle_v2_pit",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0008,
        "ic_tolerance": 0.0020,
        "sharpe_expected": -0.04,
        "sharpe_tolerance": 0.15,
        "description": "13 features, PIT — chronological Stage 2 (89% collapse on weak features)",
    },
    "stage_3_anomaly": {
        "experiment":   "extended_kaggle_v2_anomaly",
        "model_id":     "lightgbm_kaggle_v2_anomaly",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0033,
        "ic_tolerance": 0.0025,
        "sharpe_expected": +0.083,
        "sharpe_tolerance": 0.20,
        "description": "16 features (+ academic anomalies), PIT — chronological Stage 3",
    },
    "headline_baseline": {
        "experiment":   "extended_kaggle_v2_ohlcv",
        "model_id":     "lightgbm_kaggle_v2_ohlcv",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0055,
        "ic_tolerance": 0.0030,
        "sharpe_expected": +0.21,
        "sharpe_tolerance": 0.15,
        "description": "22 features (primary model), PIT, full sample",
    },
    "pit_off_ablation": {
        "experiment":   "extended_kaggle_v2_ohlcv_subset",
        "model_id":     "lightgbm_kaggle_v2_ohlcv_subset",
        "universe":     "sp500",
        "pit_filter":   False,
        "ic_expected":  +0.0142,
        "ic_tolerance": 0.0040,
        "sharpe_expected": +0.39,
        "sharpe_tolerance": 0.20,
        "description": "22 features, subset / no PIT — survivorship-bias cost on the strong feature set",
    },
    "headline_regime": {
        "experiment":   "extended_kaggle_v2_ohlcv",   # same predictions, just sliced
        "model_id":     "lightgbm_kaggle_v2_ohlcv",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "cutoff":       date(2022, 10, 10),
        "ic_expected":  +0.0183,
        "ic_tolerance": 0.0050,
        "sharpe_expected": +0.86,
        "sharpe_tolerance": 0.25,
        "description": "22 features, PIT, post-October-2022 — legacy primary (current headline is L1 +0.0695)",
    },

}

print("Expected README values:")
for stage, cfg in EXPECTED.items():
    print(f"  {stage:20s} IC = {cfg['ic_expected']:+.4f}   ({cfg['description']})")

Expected README values:
  stage_1_naive        IC = +0.0075   (13 features, subset, no PIT — chronological Stage 1 (historical))
  stage_2_pit          IC = +0.0008   (13 features, PIT — chronological Stage 2 (89% collapse on weak features))
  stage_3_anomaly      IC = +0.0033   (16 features (+ academic anomalies), PIT — chronological Stage 3)
  headline_baseline    IC = +0.0055   (22 features (primary model), PIT, full sample)
  pit_off_ablation     IC = +0.0142   (22 features, subset / no PIT — survivorship-bias cost on the strong feature set)
  headline_regime      IC = +0.0183   (22 features, PIT, post-October-2022 — legacy primary (current headline is L1 +0.0695))


## Helper functions

Two utilities:
- `fetch_predictions(model_id)` — pull the latest predictions for a model from the store, deduping on `generated_at` per `(date, ticker)`.
- `join_to_realized(preds, universe, pit_filter)` — join to the realized 5-day forward excess return panel.

In [2]:
def fetch_predictions(model_id: str) -> pl.DataFrame:
    """Pull predictions for one model_id, deduplicated to the latest generated_at."""
    store = PredictionStore(read_only=True)
    try:
        df = store.query(f"""
            WITH dedup AS (
                SELECT model_id, prediction_date, ticker, prediction,
                    ROW_NUMBER() OVER (
                        PARTITION BY model_id, prediction_date, ticker
                        ORDER BY generated_at DESC
                    ) AS rn
                FROM predictions
                WHERE model_id = '{model_id}'
            )
            SELECT prediction_date AS date, ticker, prediction
            FROM dedup
            WHERE rn = 1
        """)
    finally:
        store.close()
    return df


def join_to_realized(preds: pl.DataFrame, universe: str, pit_filter: bool) -> pl.DataFrame:
    """Join predictions to the realized 5-day forward excess return from the price panel."""
    panel = load_panel(universe=universe, start="2017-01-01", pit_filter=pit_filter)
    panel = add_forward_excess_return(panel, horizon_days=5)
    realized = panel.select("date", "ticker", pl.col("y").alias("realized"))
    return preds.join(realized, on=["date", "ticker"], how="inner")

## Verify each stage

The next four cells each run one stage's verification. Output format:

```
stage_1_baseline       PASS   IC = +0.0073  (expected +0.0075, |drift| = 0.0002 < 0.0040)
```

or:

```
stage_1_baseline       FAIL   no predictions found in store for model_id 'lightgbm_kaggle_v2'
                              -> did you run `python -m price_model.cli run -e extended_kaggle_v2` first?
```

In [3]:
def verify(stage_name: str) -> dict[str, object]:
    """Verify a single stage. Returns a result dict for the final summary."""
    cfg = EXPECTED[stage_name]
    preds = fetch_predictions(cfg["model_id"])
    if preds.height == 0:
        return {
            "stage": stage_name,
            "status": "FAIL",
            "reason": (
                f"no predictions found in store for model_id '{cfg['model_id']}'\n"
                f"  -> did you run `python -m price_model.cli run -e {cfg['experiment']}` first?"
            ),
        }

    eval_df = join_to_realized(preds, cfg["universe"], cfg["pit_filter"])
    if eval_df.height == 0:
        return {
            "stage": stage_name,
            "status": "FAIL",
            "reason": f"prediction-realized join returned 0 rows for universe '{cfg['universe']}'",
        }

    # Stages 1-3: compute full-sample IC. Stage 4: compute post-cutoff IC + Sharpe.
    if stage_name == "headline_regime":
        split = time_split_evaluate(eval_df, cutoff=cfg["cutoff"], horizon_days=5)
        post = split.metrics_b
        ic = post.information_coefficient
        sharpe = post.long_short_sharpe
        n_dates = post.n_dates
    else:
        reported = summarize(eval_df, horizon_days=5)
        ic = reported.information_coefficient
        sharpe = reported.long_short_sharpe
        n_dates = reported.n_dates

    ic_drift = abs(ic - cfg["ic_expected"])
    ic_ok = ic_drift <= cfg["ic_tolerance"]
    sharpe_ok = True
    if "sharpe_expected" in cfg:
        sharpe_drift = abs(sharpe - cfg["sharpe_expected"])
        sharpe_ok = sharpe_drift <= cfg["sharpe_tolerance"]

    return {
        "stage": stage_name,
        "status": "PASS" if (ic_ok and sharpe_ok) else "DRIFT",
        "ic_actual": ic,
        "ic_expected": cfg["ic_expected"],
        "ic_drift": ic_drift,
        "ic_tolerance": cfg["ic_tolerance"],
        "sharpe_actual": sharpe,
        "sharpe_expected": cfg.get("sharpe_expected"),
        "n_dates": n_dates,
        "description": cfg["description"],
    }


def fmt_result(r: dict) -> str:
    if r["status"] == "FAIL":
        return f"{r['stage']:20s} FAIL   {r['reason']}"
    line = (
        f"{r['stage']:20s} {r['status']:5s}  "
        f"IC = {r['ic_actual']:+.4f}  (expected {r['ic_expected']:+.4f}, "
        f"|drift| = {r['ic_drift']:.4f} {'≤' if r['ic_drift'] <= r['ic_tolerance'] else '>'}"
        f" {r['ic_tolerance']:.4f})"
    )
    if r.get("sharpe_expected") is not None:
        line += f"\n{'':20s}        Sharpe = {r['sharpe_actual']:+.3f}  (expected {r['sharpe_expected']:+.3f})"
    line += f"\n{'':20s}        n_dates = {r['n_dates']:,}  ({r['description']})"
    return line

In [4]:
# Verify all six ablation cells
results = []
for stage_name in EXPECTED:
    r = verify(stage_name)
    results.append(r)
    print(fmt_result(r))
    print()

stage_1_naive        PASS   IC = +0.0059  (expected +0.0075, |drift| = 0.0016 ≤ 0.0040)
                            Sharpe = +0.183  (expected +0.280)
                            n_dates = 1,818  (13 features, subset, no PIT — chronological Stage 1 (historical))



$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned no data for HOLX after 3 attempts
$SEE: possibly delisted; no price data found  (1d 2026-04-09 -> 2026-06-25) (Yaho

stage_2_pit          PASS   IC = +0.0008  (expected +0.0008, |drift| = 0.0000 ≤ 0.0020)
                            Sharpe = -0.052  (expected -0.040)
                            n_dates = 1,817  (13 features, PIT — chronological Stage 2 (89% collapse on weak features))



$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned no data for HOLX after 3 attempts
$SEE: possibly delisted; no price data found  (1d 2026-04-09 -> 2026-06-25) (Yaho

stage_3_anomaly      PASS   IC = +0.0032  (expected +0.0033, |drift| = 0.0001 ≤ 0.0025)
                            Sharpe = +0.059  (expected +0.083)
                            n_dates = 1,763  (16 features (+ academic anomalies), PIT — chronological Stage 3)



$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned no data for HOLX after 3 attempts
$SEE: possibly delisted; no price data found  (1d 2026-04-09 -> 2026-06-25) (Yaho

headline_baseline    PASS   IC = +0.0052  (expected +0.0055, |drift| = 0.0003 ≤ 0.0030)
                            Sharpe = +0.180  (expected +0.210)
                            n_dates = 1,763  (22 features (primary model), PIT, full sample)

pit_off_ablation     PASS   IC = +0.0146  (expected +0.0142, |drift| = 0.0004 ≤ 0.0040)
                            Sharpe = +0.412  (expected +0.390)
                            n_dates = 1,763  (22 features, subset / no PIT — survivorship-bias cost on the strong feature set)



$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-25) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned no data for HOLX after 3 attempts
$SEE: possibly delisted; no price data found  (1d 2026-04-09 -> 2026-06-25) (Yaho

headline_regime      PASS   IC = +0.0177  (expected +0.0183, |drift| = 0.0006 ≤ 0.0050)
                            Sharpe = +0.773  (expected +0.860)
                            n_dates = 910  (22 features, PIT, post-October-2022 — legacy primary (current headline is L1 +0.0695))



## Final summary

If all six ablation cells show `PASS`, the reproduction matches the README's primary estimates within reasonable yfinance drift.

If one or more stages show `DRIFT`, the prediction was generated but the numerical value moved more than the tolerance band. Most common causes:
- Your yfinance cache is significantly newer than the snapshot used to write the README (Wikipedia membership has changed; new tickers were added/removed).
- Your Ken French factor file has a different cutoff date (KF refreshes monthly).
- Hyperparameter drift if the YAML was edited.

If one or more stages show `FAIL`, the relevant experiment hasn't been run yet. Re-read the prerequisites checklist at the top.

In [5]:
status_counts = {"PASS": 0, "DRIFT": 0, "FAIL": 0}
for r in results:
    status_counts[r["status"]] += 1

print(f"Summary: {status_counts['PASS']} PASS, {status_counts['DRIFT']} DRIFT, {status_counts['FAIL']} FAIL")
if status_counts["FAIL"] > 0:
    print("\n→ At least one experiment is missing from the store. Run the CLI commands in the prerequisites checklist.")
elif status_counts["DRIFT"] > 0:
    print("\n→ All experiments produced predictions, but at least one IC drifted beyond tolerance.")
    print("  Check yfinance data freshness or KF refresh date relative to the README snapshot.")
else:
    print("\n→ All six ablation cells reproduce within tolerance. The LEGACY v2 LightGBM primary")
    print("  estimate of +0.0183 IC / Sharpe +0.86 (post-October-2022, 22 features, PIT) is")
    print("  independently verified, and the three orthogonal ablations (universe/PIT, features,")
    print("  regime) reproduce on your data.")
    print()
    print("  The CURRENT README headline is the 9-feature L1 regression at 21-day horizon")
    print("  (`lasso_elasso_pit_h21`, IC +0.0695, t = +5.25). Verify it via:")
    print("    PYTHONPATH=src python scripts/compare_apples_to_apples.py --since 2025-01-01 --horizon 21")


Summary: 6 PASS, 0 DRIFT, 0 FAIL

→ All six ablation cells reproduce within tolerance. The LEGACY v2 LightGBM primary
  estimate of +0.0183 IC / Sharpe +0.86 (post-October-2022, 22 features, PIT) is
  independently verified, and the three orthogonal ablations (universe/PIT, features,
  regime) reproduce on your data.

  The CURRENT README headline is the 9-feature L1 regression at 21-day horizon
  (`lasso_elasso_pit_h21`, IC +0.0695, t = +5.25). Verify it via:
    PYTHONPATH=src python scripts/compare_apples_to_apples.py --since 2025-01-01 --horizon 21


## Honest caveats about reproduction

Even a perfect PASS on all six ablation cells above does not mean the underlying findings — legacy or current — would survive every test:

- **Single window.** The six reported numbers are computed on the same data window and ticker universe described in the README. The current README headline (L1 regression +0.0695 IC on 2025+) is also a single-window result; a multi-window stress test has not been performed and is enumerated in the README's Discussion section as the most-needed extension.
- **Gross of transaction costs.** The legacy +0.0183 post-October-2022 IC is gross of transaction costs, taxes, and slippage; the README headline +0.0695 IC reports both gross and net 20 bp Sharpe but does not include slippage on illiquid names or capital-gains taxes. After realistic retail costs, the after-cost edge for an individual investor on either model is approximately zero.
- **Partial PIT correction.** yfinance's drop-list (~12% of historical S&P 500 members, including SIVB / FRC / SBNY) means our PIT correction is partial. A truly bias-free backtest with paid data would likely produce somewhat lower IC on both the legacy and the current headline — the bank failures would be in the cross-section.

See the README's "Data quality and methodological limitations" section for the full enumeration. Those caveats apply uniformly across all reported ICs in the README.
